# AutoML vs. Manual Pipeline: Experimento de Eficiência Computacional

**Objetivo**: Demonstrar que o uso de AutoML sem pré-processamento prévio gera  
custos computacionais (RAM, tempo) significativamente maiores comparado a um  
pipeline manual otimizado equivalente.

**Métricas coletadas**:
- Tempo total de execução (segundos)
- Pico de memória RSS (MiB)
- Área sob a curva memória×tempo (MiB·s)

---
| Parâmetro | Valor |
|-----------|-------|
| Dataset | `adult` (OpenML #1590) |
| Seed | `42` |
| Monitor interval | `100ms` |
| Python | ≥ 3.10 |

## 0. Instalação de dependências

> Execute esta célula apenas na primeira execução do ambiente.

In [ ]:
# ── Instalação das dependências do experimento ──────────────────────────────
# Descomente e execute uma única vez num ambiente limpo.

# !pip install autogluon scikit-learn pandas numpy psutil scipy \
#              matplotlib seaborn openml --quiet

## 1. Importações e Configuração Global

In [ ]:
# ── Stdlib ──────────────────────────────────────────────────────────────────
import os
import warnings
import gc
from pathlib import Path

# ── Ciência de dados ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Machine Learning ────────────────────────────────────────────────────────
import openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# ── Monitoramento ────────────────────────────────────────────────────────────
from utils import ResourceMonitor, plot_memory_comparison, build_comparison_dataframe

# ── Visualização ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# ── Supressão de warnings não-críticos ──────────────────────────────────────
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
os.environ["AUTOGLUON_VERBOSITY"] = "0"  # silencia AutoGluon

# ── Reprodutibilidade global ─────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Diretório de artefatos ───────────────────────────────────────────────────
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

print("✓ Ambiente configurado. SEED =", SEED)

## 2. Preparação do Dataset

Usamos o **Adult Income dataset** (OpenML ID 1590), um benchmark clássico com:
- 48.842 instâncias, 14 features (mistas: numéricas + categóricas)
- Target binário: `income` (≤50K / >50K)
- Valores faltantes naturais em 3 colunas

Esse dataset é ideal pois expõe claramente o comportamento do AutoML frente a  
dados sujos — ele precisa inferir e tratar tudo internamente.

In [ ]:
# ── Download via OpenML API ───────────────────────────────────────────────────
print("Baixando dataset 'Adult Income' (OpenML #1590)...")
dataset = openml.datasets.get_dataset(1590)
X_raw, y_raw, _, feature_names = dataset.get_data(
    dataset_format="dataframe",
    target=dataset.default_target_attribute,
)

print(f"Shape: {X_raw.shape} | Target: {dataset.default_target_attribute}")
print(f"Missing values:\n{X_raw.isnull().sum()[X_raw.isnull().sum() > 0]}")
X_raw.head(3)

In [ ]:
# ── Split treino / teste (70/30, estratificado) ──────────────────────────────
# O split é feito UMA VEZ e compartilhado entre os dois pipelines
# para garantir comparação justa.

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw,
    test_size=0.30,
    random_state=SEED,
    stratify=y_raw,
)

print(f"Treino: {X_train_raw.shape} | Teste: {X_test_raw.shape}")
print(f"Distribuição target (treino):\n{y_train.value_counts(normalize=True).round(3)}")

## 3. Pipeline A — AutoGluon (sem pré-processamento manual)

AutoGluon recebe os dados **exatamente como vieram do OpenML** — sem nenhum  
tratamento externo. Isso força o framework a:
1. Inferir tipos de colunas internamente
2. Tratar NaNs com sua própria lógica
3. Codificar categóricas
4. Treinar múltiplos modelos (ensemble)

> **Nota**: `presets='medium_quality'` limita o tempo mas é representativo  
> do uso real. Para experimento completo, use `'best_quality'`.

In [ ]:
from autogluon.tabular import TabularPredictor

# ── Preparação dos dados para AutoGluon (requer target junto dos features) ──
train_ag = X_train_raw.copy()
train_ag["target"] = y_train.values

test_ag = X_test_raw.copy()

# ── Monitoramento + Treinamento ───────────────────────────────────────────────
monitor_ag = ResourceMonitor(interval=0.1)

print("[Pipeline A] Iniciando treinamento AutoGluon...")
monitor_ag.start()

predictor_ag = TabularPredictor(
    label="target",
    path=str(ARTIFACTS_DIR / "autogluon_model"),
    verbosity=0,
).fit(
    train_data=train_ag,
    presets="medium_quality",
    time_limit=300,          # 5 minutos máximo
    ag_args_fit={"num_cpus": 4},
)

monitor_ag.stop()
print("[Pipeline A] Treinamento concluído.")

# ── Avaliação ─────────────────────────────────────────────────────────────────
y_pred_ag = predictor_ag.predict(test_ag)
acc_ag = accuracy_score(y_test, y_pred_ag)

print(f"\n{'='*50}")
print(f"[Pipeline A] Acurácia: {acc_ag:.4f}")
print(f"[Pipeline A] Métricas de recursos:")
for k, v in monitor_ag.summary().items():
    print(f"  {k:30s}: {v}")

# Salvar amostras brutas
monitor_ag.to_csv(ARTIFACTS_DIR / "monitor_autogluon.csv")

## 4. Pipeline B — Scikit-Learn Manual Otimizado

O pipeline manual segue boas práticas de Engenharia de Features:

1. **Imputação**: mediana (numéricas) e moda (categóricas)
2. **Encoding**: OrdinalEncoder para GBM (evita explosão de dimensionalidade)
3. **Scaling**: StandardScaler nas features numéricas
4. **Redução**: PCA retendo 95% da variância (opcional, demonstra o conceito)
5. **Modelo**: GradientBoostingClassifier — comparável ao ensemble do AutoGluon

> O modelo é intencionalmente equivalente em capacidade ao melhor modelo  
> individualmente produzido pelo AutoGluon, para isolar o efeito do  
> pré-processamento na eficiência.

In [ ]:
# ── Identificação de tipos de colunas ────────────────────────────────────────
numeric_cols = X_train_raw.select_dtypes(include=["number"]).columns.tolist()
cat_cols     = X_train_raw.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Features numéricas ({len(numeric_cols)}): {numeric_cols}")
print(f"Features categóricas ({len(cat_cols)}): {cat_cols}")

In [ ]:
# ── Construção do pipeline scikit-learn ──────────────────────────────────────

# Sub-pipeline para features numéricas
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

# Sub-pipeline para features categóricas
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )),
])

# Pré-processador unificado
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop",
    n_jobs=-1,
)

# Pipeline completo com PCA + GBM
pipeline_manual = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("pca", PCA(n_components=0.95, random_state=SEED)),   # retém 95% da variância
    ("classifier", GradientBoostingClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        random_state=SEED,
        n_iter_no_change=10,
        validation_fraction=0.1,
    )),
])

print("Pipeline manual construído:")
print(pipeline_manual)

In [ ]:
# ── Encoding do target (sklearn exige labels consistentes) ───────────────────
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

# ── Limpeza de memória antes de medir ────────────────────────────────────────
gc.collect()

# ── Monitoramento + Treinamento ───────────────────────────────────────────────
monitor_sk = ResourceMonitor(interval=0.1)

print("[Pipeline B] Iniciando treinamento Manual Sklearn...")
monitor_sk.start()

pipeline_manual.fit(X_train_raw, y_train_enc)

monitor_sk.stop()
print("[Pipeline B] Treinamento concluído.")

# ── Avaliação ─────────────────────────────────────────────────────────────────
y_pred_sk = pipeline_manual.predict(X_test_raw)
acc_sk = accuracy_score(y_test_enc, y_pred_sk)

print(f"\n{'='*50}")
print(f"[Pipeline B] Acurácia: {acc_sk:.4f}")
print(f"[Pipeline B] Métricas de recursos:")
for k, v in monitor_sk.summary().items():
    print(f"  {k:30s}: {v}")

# Salvar amostras brutas
monitor_sk.to_csv(ARTIFACTS_DIR / "monitor_sklearn.csv")

## 5. Comparação — DataFrame de Resultados

In [ ]:
# ── DataFrame de comparação final ────────────────────────────────────────────
results = [
    {"Pipeline": "A — AutoGluon (raw)",  "monitor": monitor_ag, "accuracy": acc_ag},
    {"Pipeline": "B — Manual (sklearn)", "monitor": monitor_sk, "accuracy": acc_sk},
]

df_comparison = build_comparison_dataframe(results)

# Colunas de razão (A / B) para destacar o custo relativo
row_ag = df_comparison[df_comparison["Pipeline"].str.startswith("A")].iloc[0]
row_sk = df_comparison[df_comparison["Pipeline"].str.startswith("B")].iloc[0]

ratios = pd.Series({
    "Razão_Tempo":   row_ag["Tempo_Total_s"]       / row_sk["Tempo_Total_s"],
    "Razão_Pico":    row_ag["Pico_Memoria_MB"]     / row_sk["Pico_Memoria_MB"],
    "Razão_AUC_Mem": row_ag["Area_Sob_Curva_MiBs"] / row_sk["Area_Sob_Curva_MiBs"],
}, name="AutoGluon / Manual")

print("\n📊 RESULTADOS FINAIS")
print("=" * 70)
display(df_comparison.set_index("Pipeline"))

print("\n📈 RAZÕES DE CUSTO  (AutoGluon ÷ Manual)")
print("=" * 70)
display(ratios.to_frame())

# Exportar para CSV
df_comparison.to_csv(ARTIFACTS_DIR / "comparison_results.csv", index=False)
print(f"\n✓ Resultados salvos em {ARTIFACTS_DIR / 'comparison_results.csv'}")

## 6. Visualização — Memory vs. Time (Estilo IEEE/ACM)

In [ ]:
# ── Gráfico 1: Memory × Time (linha temporal) ────────────────────────────────
fig_mem = plot_memory_comparison(
    monitors={
        "Pipeline A: AutoGluon (raw)": monitor_ag,
        "Pipeline B: Manual (sklearn)": monitor_sk,
    },
    title="Memory Consumption During Model Training",
    output_path=str(ARTIFACTS_DIR / "fig_memory_time.pdf"),
    figsize=(7.16, 4.5),  # largura de 2 colunas IEEE
)
plt.show()

In [ ]:
# ── Gráfico 2: Barras de comparação com múltiplas métricas ───────────────────
import matplotlib

IEEE_STYLE = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "grid.alpha": 0.5,
    "figure.dpi": 300,
}

METRICS = ["Tempo_Total_s", "Pico_Memoria_MB", "Area_Sob_Curva_MiBs", "Acuracia"]
LABELS  = ["Total Time (s)", "Peak Memory (MiB)", "Mem·Time AUC (MiB·s)", "Accuracy"]
COLORS  = ["#E64B35", "#4DBBD5"]

with matplotlib.rc_context(IEEE_STYLE):
    fig, axes = plt.subplots(1, 4, figsize=(7.16, 3.0))
    fig.suptitle("Pipeline Comparison: AutoGluon vs. Manual (sklearn)",
                 fontsize=10, y=1.02)

    for ax, metric, label in zip(axes, METRICS, LABELS):
        vals = df_comparison[metric].values
        pipes = ["AutoGluon", "Manual"]
        bars = ax.bar(pipes, vals, color=COLORS, edgecolor="black",
                      linewidth=0.6, width=0.5)

        # Valor no topo da barra
        for bar, v in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02,
                f"{v:.2f}",
                ha="center", va="bottom", fontsize=7,
            )

        ax.set_title(label, pad=4)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["A\nAutoGluon", "B\nManual"], fontsize=7)
        ax.set_ylim(0, max(vals) * 1.25)
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(4))

    fig.tight_layout()
    fig.savefig(ARTIFACTS_DIR / "fig_bar_comparison.pdf",
                dpi=300, bbox_inches="tight")
    plt.show()

print(f"✓ Gráfico de barras salvo em {ARTIFACTS_DIR / 'fig_bar_comparison.pdf'}")

In [ ]:
# ── Gráfico 3: Área sob a curva destacada (fill_between) ─────────────────────
df_ag = monitor_ag.to_dataframe()
df_sk = monitor_sk.to_dataframe()

with matplotlib.rc_context(IEEE_STYLE):
    fig, ax = plt.subplots(figsize=(7.16, 4.0))

    ax.fill_between(df_ag["timestamp_s"], df_ag["rss_mb"],
                    alpha=0.25, color="#E64B35", label="_nolegend_")
    ax.fill_between(df_sk["timestamp_s"], df_sk["rss_mb"],
                    alpha=0.25, color="#4DBBD5", label="_nolegend_")

    ax.plot(df_ag["timestamp_s"], df_ag["rss_mb"],
            color="#E64B35", lw=1.8,
            label=f"A: AutoGluon  (AUC={monitor_ag.memory_time_auc():.0f} MiB·s)")
    ax.plot(df_sk["timestamp_s"], df_sk["rss_mb"],
            color="#4DBBD5", lw=1.8, ls="--",
            label=f"B: Manual     (AUC={monitor_sk.memory_time_auc():.0f} MiB·s)")

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Resident Set Size (MiB)")
    ax.set_title("Memory-Time Integral (Shaded Area = Computational Cost)", pad=8)
    ax.legend(loc="upper right")
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(4))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(4))

    fig.tight_layout()
    fig.savefig(ARTIFACTS_DIR / "fig_auc_comparison.pdf",
                dpi=300, bbox_inches="tight")
    plt.show()

print(f"✓ Gráfico AUC salvo em {ARTIFACTS_DIR / 'fig_auc_comparison.pdf'}")

## 7. Análise e Conclusão

In [ ]:
# ── Resumo executivo ──────────────────────────────────────────────────────────
print("\n" + "="*70)
print("  RESUMO DO EXPERIMENTO")
print("="*70)
print(f"  Dataset : Adult Income (OpenML #1590)")
print(f"  Treino  : {X_train_raw.shape[0]:,} amostras | Teste: {X_test_raw.shape[0]:,} amostras")
print(f"  Seed    : {SEED}")
print()
print(f"  {'Métrica':<35} {'AutoGluon':>12} {'Manual':>12} {'Razão A/B':>10}")
print(f"  {'-'*69}")

metrics_map = [
    ("Tempo Total (s)",         "Tempo_Total_s"),
    ("Pico de Memória (MiB)",   "Pico_Memoria_MB"),
    ("AUC Memória (MiB·s)",     "Area_Sob_Curva_MiBs"),
    ("Acurácia",                 "Acuracia"),
]

for name, col in metrics_map:
    v_ag = row_ag[col]
    v_sk = row_sk[col]
    ratio = v_ag / v_sk if v_sk != 0 else float("inf")
    print(f"  {name:<35} {v_ag:>12.2f} {v_sk:>12.2f} {ratio:>10.2f}×")

print("="*70)
print()
print("  Conclusão:")
print(f"  O Pipeline A (AutoGluon) consumiu {ratios['Razão_AUC_Mem']:.1f}× mais")
print(f"  'custo de memória integrado' e foi {ratios['Razão_Tempo']:.1f}× mais lento,")
print(f"  com acurácia {'superior' if acc_ag > acc_sk else 'inferior ou equivalente'} ao pipeline manual.")
print()
print("  Arquivos gerados:")
for f in sorted(ARTIFACTS_DIR.iterdir()):
    print(f"    {f}")

---
## Apêndice: Snippet reutilizável para monitoramento

```python
from utils import ResourceMonitor

# Uso como context manager (recomendado)
monitor = ResourceMonitor(interval=0.1)
with monitor:
    model.fit(X_train, y_train)

print(monitor.summary())
df = monitor.to_dataframe()   # pandas DataFrame com timestamp_s, rss_mb
auc = monitor.memory_time_auc()  # MiB·s
```

```python
# Uso como decorator
from utils import monitor_resources

@monitor_resources(interval=0.05)
def meu_treino():
    model.fit(X, y)

result, mon = meu_treino()
```